# Import Libraries

In [201]:
import pandas as pd
import numpy as np
import spacy
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import BatchNormalization, SimpleRNN, GlobalAveragePooling1D, Bidirectional, MaxPooling1D, Flatten, Layer
from tensorflow.keras.layers import Dense, LSTM, Input, Dropout, GlobalMaxPooling1D, Conv1D, GRU, Bidirectional, Lambda, Attention
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from tensorflow.keras.metrics import F1Score
from tensorflow.keras.optimizers import Adam
from gensim.models import Word2Vec
from transformers import BertTokenizer, TFBertModel
import tensorflow as tf

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full content in each cell
pd.set_option('display.width', 1000)  # Set max width

# Load spaCy's English model
nlp = spacy.load('en_core_web_sm')

In [202]:
def preprocess_text(text):
    # Define interrogative words to KEEP
    interrogatives = {"what", "why", "how", "who", "where", "when", "which", "whom", "whose", "no", "not",
                    "very" ,"too" ,"too" ,"just", "if", "but", "however", "without", "like"}
    custom_stopwords = set(nlp.Defaults.stop_words)
    custom_stopwords -= interrogatives

    doc = nlp(text.lower().strip())  # Lowercase and remove whitespace
    
# Process tokens: lemmatize, filter stopwords/punct/numbers, keep interrogatives
    tokens = [
        token.lemma_ 
        for token in doc 
        if (
            (not token.is_stop or token.text in interrogatives) and  # Keep interrogatives
            not token.is_punct and token.is_alpha                                  # Remove punctuation
            # (token.is_alpha or token.like_num)                       # Keep words/numbers
        )
    ]

    return ' '.join(tokens)

In [203]:
class AttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal")
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros")
        super().build(input_shape)

    def call(self, x):
        e = tf.keras.backend.tanh(tf.keras.backend.dot(x, self.W) + self.b)
        a = tf.keras.backend.softmax(e, axis=1)
        output = x * a
        return output

In [204]:
# Tokenize input text
# Load BERT tokenizer and model
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
bert_model = TFBertModel.from_pretrained(model_name)

def tokenize_texts(texts, max_len):
    encodings = tokenizer(
        texts.tolist(),
        max_length=max_len,
        truncation=True,
        padding='max_length',
        return_tensors='tf'
    )

    outputs = bert_model(encodings)

    return outputs.last_hidden_state

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

In [205]:
def make_dataset(texts, labels, tokenizer, shuffle=False, max_len= 20, batch_size = 32):
    def gen():
        for t, l in zip(texts, labels):
            enc = tokenizer(
                t,
                truncation=True,
                padding='max_length',
                max_length=max_len,
                return_tensors='tf'
            )
            yield ({'input_ids': enc['input_ids'][0], 'attention_mask': enc['attention_mask'][0]}, l)

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            {'input_ids': tf.TensorSpec(shape=(max_len,), dtype=tf.int32),
             'attention_mask': tf.TensorSpec(shape=(max_len,), dtype=tf.int32)},
            tf.TensorSpec(shape=(), dtype=tf.int32)
        )
    )
    if shuffle:
        ds = ds.shuffle(buffer_size=len(texts))
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Pre-Processing

### Import Data

In [206]:
mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

# Load dataset
df = pd.DataFrame()
for i in [2,3,4,5]:
    q_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset' + str(i) + '.csv')
    q_df['dataset_id'] = i
    df = pd.concat([df , q_df], )
    
df = df.reset_index(drop=True)

# Apply preprocessing
df['label'] = df['label'].str.lower()
df['label'] = df['label'].replace(mapping)

max_len = max(len(tokenizer.encode(text, add_special_tokens=True)) for text in df['question'])
print("Max sequence length:", max_len)

Max sequence length: 95


In [157]:
train_q, test_q, train_label, test_label = train_test_split(df['question'], df['label'], test_size= 0.2, stratify= df['label'])

In [207]:
mapping = {
    'knowledge': 'knowledge',
    'remember': 'knowledge',
    'comprehension': 'comprehension',
    'understand': 'comprehension',
    'application': 'application',
    'apply': 'application',
    'analysis': 'analysis',
    'analyse': 'analysis',
    'evaluation': 'evaluation',
    'evaluate': 'evaluation',
    'synthesis': 'synthesis',
    'create': 'synthesis'
}

# Load dataset
test_df = pd.read_csv(os.getcwd().replace('notebook' , 'dataset') + '/dataset' + str(1) + '.csv')
    
test_df = test_df.reset_index(drop=True)

# Apply preprocessing
test_df['label'] = test_df['label'].str.lower()
test_df['label'] = test_df['label'].replace(mapping)

train_q, train_label = df['question'], df['label']
test_q, test_label = test_df['question'], test_df['label']


## Tokenize

### BERT

In [208]:
# Parameters
num_classes = 6

# Embedding
x_train = tokenize_texts(train_q, max_len)
x_test = tokenize_texts(test_q, max_len)

In [209]:
y_mapper = {
    'knowledge' : 0,
    'comprehension' : 1,
    'application' : 2,
    'analysis' : 3,
    'synthesis' : 4,
    'evaluation' : 5
}

y_mapped = train_label.map(y_mapper)
y_test_mapped = test_label.map(y_mapper)

y_train = to_categorical(np.asarray(y_mapped))
y_test = to_categorical(np.asarray(y_test_mapped))

# Modelling

In [210]:
callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
]


## RNN

In [212]:
rnn_model = Sequential([
    Input(shape=(max_len, 768)),
    SimpleRNN(128,return_sequences=True, recurrent_dropout=0.1),
    GlobalAveragePooling1D(),
    Dropout(0.2),
    Dense(64, activation='swish'),
    Dense(6, activation='softmax')
])


rnn_model.compile(loss='categorical_crossentropy', 
                   optimizer=Adam(learning_rate=1e-4), 
                   metrics=['accuracy',
                            F1Score(average='macro', name='f1_macro')]
                            )

rnn_model.summary()

Model: "sequential_45"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_37 (SimpleRNN)       │ (None, 95, 128)        │       114,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_26     │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_142 (Dropout)           │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_148 (Dense)               │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_149 (Dense)               │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 123,462 (482.27 KB)

 Trainable params: 123,462 (482.27 KB)

 Non-trainable params: 0 (0.00 B)

In [213]:
rnn_history = rnn_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.2, callbacks=callbacks)

Epoch 1/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.3226 - f1_macro: 0.1251 - loss: 1.6932 - val_accuracy: 0.6338 - val_f1_macro: 0.1293 - val_loss: 1.2882 - learning_rate: 1.0000e-04
Epoch 2/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.3921 - f1_macro: 0.1333 - loss: 1.5650 - val_accuracy: 0.6390 - val_f1_macro: 0.1592 - val_loss: 1.1638 - learning_rate: 1.0000e-04
Epoch 3/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.4464 - f1_macro: 0.2112 - loss: 1.4150 - val_accuracy: 0.6727 - val_f1_macro: 0.3152 - val_loss: 1.0843 - learning_rate: 1.0000e-04
Epoch 4/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.5036 - f1_macro: 0.3617 - loss: 1.2997 - val_accuracy: 0.6883 - val_f1_macro: 0.4041 - val_loss: 0.9851 - learning_rate: 1.0000e-04
Epoch 5/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.5723 - f1_macro: 0.4600 - loss: 1.1594 - val_accuracy: 0.7221 - val_f1_macro: 0.5009 - val_loss: 0.9054 - learning_rate: 1.0000e-04
Epoch 6/20

In [214]:
history_dict = rnn_history.history

val_acc = history_dict['val_accuracy'][-20:]
val_f1 = history_dict['val_f1_macro'][-20:]

print(f"Val Accuracy: max={np.max(val_acc):.4f}, min={np.min(val_acc):.4f}, avg={np.mean(val_acc):.4f}")
print(f"Val F1:       max={np.max(val_f1):.4f}, min={np.min(val_f1):.4f}, avg={np.mean(val_f1):.4f}")

Val Accuracy: max=0.7558, min=0.6727, avg=0.7113
Val F1:       max=0.6283, min=0.3152, avg=0.5696


In [215]:
results = rnn_model.evaluate(x_test, y_test, verbose=0)

loss, acc, f1_macro = results
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Test F1 Macro: {f1_macro:.4f}")

Test Loss: 1.4818
Test Accuracy: 45.67%
Test F1 Macro: 0.4569


## LSTM

In [216]:
lstm_model = Sequential([
    Input(shape=(max_len, 768)),
    LSTM(128, return_sequences=True, recurrent_dropout=0.1),
    GlobalAveragePooling1D(),
    Dropout(0.2),
    Dense(64, activation='swish'),
    Dense(6, activation='softmax')
])

lstm_model.compile(loss='categorical_crossentropy', 
                   optimizer=Adam(learning_rate=1e-4), 
                   metrics=['accuracy',
                            F1Score(average='macro', name='f1_macro')]
                            )

lstm_model.summary()

Model: "sequential_46"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_17 (LSTM)                  │ (None, 95, 128)        │       459,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_27     │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_143 (Dropout)           │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_150 (Dense)               │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_151 (Dense)               │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 467,910 (1.78 MB)

 Trainable params: 467,910 (1.78 MB)

 Non-trainable params: 0 (0.00 B)

In [217]:
lstm_history = lstm_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.2, callbacks=callbacks)

Epoch 1/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 7s 122ms/step - accuracy: 0.3083 - f1_macro: 0.1162 - loss: 1.7051 - val_accuracy: 0.6338 - val_f1_macro: 0.1293 - val_loss: 1.2705 - learning_rate: 1.0000e-04
Epoch 2/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 6s 116ms/step - accuracy: 0.3920 - f1_macro: 0.0952 - loss: 1.5459 - val_accuracy: 0.6442 - val_f1_macro: 0.1693 - val_loss: 1.1727 - learning_rate: 1.0000e-04
Epoch 3/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 6s 115ms/step - accuracy: 0.4687 - f1_macro: 0.2663 - loss: 1.3634 - val_accuracy: 0.6987 - val_f1_macro: 0.4438 - val_loss: 1.0268 - learning_rate: 1.0000e-04
Epoch 4/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 5s 109ms/step - accuracy: 0.5809 - f1_macro: 0.4789 - loss: 1.1622 - val_accuracy: 0.7299 - val_f1_macro: 0.5250 - val_loss: 0.9017 - learning_rate: 1.0000e-04
Epoch 5/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 5s 108ms/step - accuracy: 0.6324 - f1_macro: 0.5766 - loss: 1.0172 - val_accuracy: 0.7169 - val_f1_macro: 0.5465 - val_loss: 0.8537 - learning_rate: 1.0000e-04
Epoch

In [218]:
history_dict = lstm_history.history

val_acc = history_dict['val_accuracy'][-10:]
val_f1 = history_dict['val_f1_macro'][-10:]

print(f"Val Accuracy: max={np.max(val_acc):.4f}, min={np.min(val_acc):.4f}, avg={np.mean(val_acc):.4f}")
print(f"Val F1:       max={np.max(val_f1):.4f}, min={np.min(val_f1):.4f}, avg={np.mean(val_f1):.4f}")


Val Accuracy: max=0.7584, min=0.7065, avg=0.7327
Val F1:       max=0.6158, min=0.5736, avg=0.5998


In [219]:
results = lstm_model.evaluate(x_test, y_test, verbose=0)

loss, acc, f1_macro = results
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Test F1 Macro: {f1_macro:.4f}")

Test Loss: 1.5300
Test Accuracy: 45.33%
Test F1 Macro: 0.4475


### Bi-GRU

In [220]:
gru_model = Sequential([
    Input(shape=(max_len, 768)),
    Bidirectional(GRU(128, return_sequences=True, recurrent_dropout=0.1)),
    GlobalAveragePooling1D(),
    Dropout(0.2),
    Dense(64, activation='swish'),
    Dense(6, activation='softmax')
])

gru_model.compile(loss='categorical_crossentropy', 
                   optimizer=Adam(learning_rate=1e-4), 
                   metrics=['accuracy',
                            F1Score(average='macro', name='f1_macro')]
                            )

gru_model.summary()


Model: "sequential_47"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_32                │ (None, 95, 256)        │       689,664 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_28     │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_144 (Dropout)           │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_152 (Dense)               │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_153 (Dense)               │ (None, 6)              │           390 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 706,502 (2.70 MB)

 Trainable params: 706,502 (2.70 MB)

 Non-trainable params: 0 (0.00 B)

In [221]:
gru_history = gru_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.2, callbacks=callbacks)

Epoch 1/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 169ms/step - accuracy: 0.3756 - f1_macro: 0.1045 - loss: 1.6271 - val_accuracy: 0.6390 - val_f1_macro: 0.1522 - val_loss: 1.2745 - learning_rate: 1.0000e-04
Epoch 2/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 8s 168ms/step - accuracy: 0.4460 - f1_macro: 0.1915 - loss: 1.4245 - val_accuracy: 0.6701 - val_f1_macro: 0.2867 - val_loss: 1.1398 - learning_rate: 1.0000e-04
Epoch 3/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 12s 236ms/step - accuracy: 0.5130 - f1_macro: 0.3569 - loss: 1.2946 - val_accuracy: 0.7091 - val_f1_macro: 0.3941 - val_loss: 1.0178 - learning_rate: 1.0000e-04
Epoch 4/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 12s 238ms/step - accuracy: 0.5748 - f1_macro: 0.4485 - loss: 1.1549 - val_accuracy: 0.7403 - val_f1_macro: 0.5281 - val_loss: 0.9610 - learning_rate: 1.0000e-04
Epoch 5/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 12s 251ms/step - accuracy: 0.6222 - f1_macro: 0.5355 - loss: 1.0554 - val_accuracy: 0.7532 - val_f1_macro: 0.5938 - val_loss: 0.8895 - learning_rate: 1.0000e-04
E

In [222]:
history_dict = gru_history.history

val_acc = history_dict['val_accuracy'][-10:]
val_f1 = history_dict['val_f1_macro'][-10:]

print(f"Val Accuracy: max={np.max(val_acc):.4f}, min={np.min(val_acc):.4f}, avg={np.mean(val_acc):.4f}")
print(f"Val F1:       max={np.max(val_f1):.4f}, min={np.min(val_f1):.4f}, avg={np.mean(val_f1):.4f}")

Val Accuracy: max=0.7403, min=0.6961, avg=0.7161
Val F1:       max=0.6396, min=0.6119, avg=0.6281


In [223]:
results = gru_model.evaluate(x_test, y_test, verbose=0)

loss, acc, f1_macro = results
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Test F1 Macro: {f1_macro:.4f}")

Test Loss: 1.4810
Test Accuracy: 48.17%
Test F1 Macro: 0.4855


### AM-BiGRU-CNN

In [224]:
# Input
inputs = Input(shape=(max_len, 768))

# 1. Attention Layer
attn_out = AttentionLayer()(inputs)

# 2. BiGRU
bigru = Bidirectional(GRU(128, return_sequences=True, recurrent_dropout=0.1))(attn_out)
drop = Dropout(0.2)(bigru)


# 3. CNN
conv = Conv1D(filters=64, kernel_size=3, activation="gelu", padding="same")(bigru)
pool = GlobalMaxPooling1D()(conv)
drop = Dropout(0.2)(pool)

# 4. Dense
dense = Dense(32, activation="swish")(drop)
drop = Dropout(0.2)(dense)
outputs = Dense(num_classes, activation="softmax")(drop)

cbgat_model = Model(inputs=inputs, outputs=outputs)
cbgat_model.compile(loss='categorical_crossentropy', 
                   optimizer=Adam(learning_rate=1e-4), 
                   metrics=['accuracy',
                            F1Score(average='macro', name='f1_macro')]
                            )
cbgat_model.summary()


Model: "functional_58"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_59 (InputLayer)     │ (None, 95, 768)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_layer_6               │ (None, 95, 768)        │           863 │
│ (AttentionLayer)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_33                │ (None, 95, 256)        │       689,664 │
│ (Bidirectional)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_18 (Conv1D)              │ (None, 95, 64)         │        49,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_7          │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_146 (Dropout)           │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_154 (Dense)               │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_147 (Dropout)           │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_155 (Dense)               │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 742,021 (2.83 MB)

 Trainable params: 742,021 (2.83 MB)

 Non-trainable params: 0 (0.00 B)

In [225]:
cbgat_history = cbgat_model.fit(x_train, y_train, epochs=200, batch_size = 32, validation_split=0.2, callbacks=callbacks)

Epoch 1/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 11s 197ms/step - accuracy: 0.3564 - f1_macro: 0.1145 - loss: 1.7804 - val_accuracy: 0.6338 - val_f1_macro: 0.1293 - val_loss: 1.6695 - learning_rate: 1.0000e-04
Epoch 2/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 197ms/step - accuracy: 0.4047 - f1_macro: 0.0960 - loss: 1.7031 - val_accuracy: 0.6338 - val_f1_macro: 0.1293 - val_loss: 1.3678 - learning_rate: 1.0000e-04
Epoch 3/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 14s 283ms/step - accuracy: 0.4001 - f1_macro: 0.0952 - loss: 1.6224 - val_accuracy: 0.6338 - val_f1_macro: 0.1293 - val_loss: 1.3180 - learning_rate: 1.0000e-04
Epoch 4/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 14s 287ms/step - accuracy: 0.4208 - f1_macro: 0.0986 - loss: 1.5973 - val_accuracy: 0.6338 - val_f1_macro: 0.1293 - val_loss: 1.3245 - learning_rate: 1.0000e-04
Epoch 5/200
49/49 ━━━━━━━━━━━━━━━━━━━━ 14s 285ms/step - accuracy: 0.3964 - f1_macro: 0.0946 - loss: 1.6056 - val_accuracy: 0.6338 - val_f1_macro: 0.1293 - val_loss: 1.2589 - learning_rate: 1.0000e-04


In [226]:
history_dict = cbgat_history.history

val_acc = history_dict['val_accuracy'][-10:]
val_f1 = history_dict['val_f1_macro'][-10:]

print(f"Val Accuracy: max={np.max(val_acc):.4f}, min={np.min(val_acc):.4f}, avg={np.mean(val_acc):.4f}")
print(f"Val F1:       max={np.max(val_f1):.4f}, min={np.min(val_f1):.4f}, avg={np.mean(val_f1):.4f}")


Val Accuracy: max=0.7792, min=0.7636, avg=0.7688
Val F1:       max=0.6846, min=0.6561, avg=0.6679


In [227]:
results = cbgat_model.evaluate(x_test, y_test, verbose=0)

loss, acc, f1_macro = results
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Test F1 Macro: {f1_macro:.4f}")

Test Loss: 1.3381
Test Accuracy: 52.67%
Test F1 Macro: 0.5239
